# Complaint2Concern

A side project comparing structured intake data, TF-IDF and frozen MiniLM text embeddings for acuity prediction on synthetic Kaggle Triagegeist data. This notebook is the canonical analysis. See the README for execution and the generated results report for measured outcomes.

## Data understanding and leakage guardrails

This is a triage-time task. All model inputs must be available at triage decision time.

- Allowed: intake structured variables, chief complaint text, prior history available at triage.
- Excluded as leakage: post-triage outcomes (for example `disposition`, `ed_los_hours`).

Missingness is handled with model pipelines (median/mode imputation in structured preprocessing).

In [ ]:
import os
import re
import sys
import time
import zipfile
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

from scipy.sparse import csr_matrix, hstack
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import GroupKFold, StratifiedKFold
from sklearn.preprocessing import LabelEncoder

try:
    import matplotlib.pyplot as plt
    MATPLOTLIB_OK = True
except Exception:
    MATPLOTLIB_OK = False


def _find_code_root() -> Path:
    # Kaggle path: attach a dataset containing triagegeist_code_assets.zip.
    kaggle_input = Path('/kaggle/input')
    if kaggle_input.exists():
        # Support either an uploaded zip or a dataset that already exposes src/configs.
        for candidate in kaggle_input.glob('*'):
            if (candidate / 'src').exists() and (candidate / 'configs').exists():
                return candidate

        zip_matches = sorted(kaggle_input.rglob('triagegeist_code_assets.zip'))
        if zip_matches:
            extract_dir = Path('/kaggle/working/triagegeist_code_assets')
            extract_dir.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(zip_matches[0], 'r') as zf:
                zf.extractall(extract_dir)
            if (extract_dir / 'src').exists() and (extract_dir / 'configs').exists():
                return extract_dir
            for nested in extract_dir.glob('*'):
                if (nested / 'src').exists() and (nested / 'configs').exists():
                    return nested

    # Local fallback: original repo layout.
    for candidate in [Path.cwd(), Path.cwd().parent]:
        if (candidate / 'src').exists() and (candidate / 'configs').exists():
            return candidate

    raise RuntimeError(
        'Could not locate project code. On Kaggle, add a dataset containing '
        'triagegeist_code_assets.zip with src/, configs/, and requirements.txt.'
    )


def _find_kaggle_data_dir() -> Path | None:
    kaggle_input = Path('/kaggle/input')
    if not kaggle_input.exists():
        return None

    candidates = []
    for candidate in kaggle_input.glob('*'):
        if (candidate / 'train.csv').exists() and (candidate / 'test.csv').exists():
            score = int((candidate / 'chief_complaints.csv').exists()) + int((candidate / 'patient_history.csv').exists())
            candidates.append((score, candidate))

    if not candidates:
        return None
    return sorted(candidates, reverse=True)[0][1]


PROJECT_ROOT = _find_code_root()
sys.path.insert(0, str(PROJECT_ROOT))
KAGGLE_DATA_DIR = _find_kaggle_data_dir()

print('PROJECT_ROOT:', PROJECT_ROOT)
print('KAGGLE_DATA_DIR:', KAGGLE_DATA_DIR)

import json, hashlib, platform
from datetime import datetime, timezone
import importlib.metadata
import torch
torch.set_num_threads(4)


In [ ]:
from src.config import load_config
from src.data_io import load_train_test
from src.embeddings import encode_complaints
from src.features_structured import fit_transform_structured, infer_structured_columns
from src.features_text import fit_transform_tfidf
from src.preprocess import preprocess_text_series
from src.utils import make_classifier, set_seed
from src.evaluate import evaluate_acuity
from src.provenance import export_run


In [ ]:
# Config
if KAGGLE_DATA_DIR is None and Path('/kaggle/input').exists():
    raise FileNotFoundError('Competition data not found in Kaggle inputs')
cfg = load_config(PROJECT_ROOT / 'configs/minilm_catboost.yaml')

if KAGGLE_DATA_DIR is not None:
    cfg['data']['kaggle_input_dir'] = str(KAGGLE_DATA_DIR)
# Optional Kaggle path override example:
# cfg['data']['kaggle_input_dir'] = '/kaggle/input/triagegeist'

DEV_SAMPLE_N = int(os.environ.get('TRIAGE_SAMPLE_N', '0')) or None  # Set an integer like 20000 for quick local iteration.
SEED = cfg['train'].get('seed', 42)
CV_MODE = 'stratified'  # 'stratified' or 'group_site'
N_SPLITS = int(os.environ.get('TRIAGE_SPLITS', '5'))
RUN_PROJECTIONS = os.environ.get('TRIAGE_PROJECTIONS', '0') == '1'
RUN_SUBMISSION = os.environ.get('TRIAGE_SUBMISSION', '0') == '1'
RUN_DIR = Path(os.environ.get('TRIAGE_RUN_DIR', str(PROJECT_ROOT / 'outputs' / 'latest')))
RUN_DIR.mkdir(parents=True, exist_ok=True)
set_seed(SEED)

# merges original train and test to complaint text by patient id
train_df, test_df = load_train_test(cfg)
if DEV_SAMPLE_N is not None and DEV_SAMPLE_N < len(train_df):
    train_df = train_df.sample(DEV_SAMPLE_N, random_state=SEED).reset_index(drop=True)

id_col = cfg['data']['id_col']
target_col = cfg['data']['target_col']
text_col = cfg['data']['text_col']
leakage_cols = cfg['data'].get('leakage_columns', [])

print('train shape:', train_df.shape)
print('test shape :', test_df.shape)
print('target col :', target_col)
print('text col   :', text_col)
print('null text train:', round(train_df[text_col].isna().mean(), 4))
print('null text test :', round(test_df[text_col].isna().mean(), 4))
print('CV mode:', CV_MODE, '| folds:', N_SPLITS)


In [ ]:
# Triage-focused EDA
print('Acuity distribution (train):')
display(train_df[target_col].value_counts(normalize=True).sort_index())

missingness = train_df.isna().mean().sort_values(ascending=False)
display(missingness.head(15).to_frame('missing_rate'))

# Vitals by acuity (median)
vitals = [c for c in ['heart_rate','respiratory_rate','temperature_c','spo2','systolic_bp','diastolic_bp','news2_score'] if c in train_df.columns]
if vitals:
    display(train_df.groupby(target_col)[vitals].median())

if MATPLOTLIB_OK:
    ax = train_df[target_col].value_counts().sort_index().plot(kind='bar', title='ESI/Triage Acuity Class Distribution')
    ax.set_xlabel('triage_acuity')
    ax.set_ylabel('count')
    plt.show()

## Shared text preprocessing

Missing text becomes an empty string. Abbreviations are expanded case-insensitively before punctuation and whitespace cleanup. Training, analysis and inference use the same functions in `src/preprocess.py`.

In [ ]:
train_df[text_col] = preprocess_text_series(train_df[text_col])
test_df[text_col] = preprocess_text_series(test_df[text_col])
print('Applied shared missing-text and abbreviation normalisation.')


In [ ]:
# Prepare split (cross-validation setup)
label_encoder = LabelEncoder()
y_all = label_encoder.fit_transform(train_df[target_col].astype(str))


def make_splitter(df, y, mode='stratified', n_splits=5, seed=42):
    if mode == 'group_site' and 'site_id' in df.columns:
        gkf = GroupKFold(n_splits=n_splits)
        return list(gkf.split(df, y, groups=df['site_id']))
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    return list(skf.split(df, y))


splits = make_splitter(train_df, y_all, mode=CV_MODE, n_splits=N_SPLITS, seed=SEED)
print(f"Built {len(splits)} CV folds")
print("y_all:", Counter(y_all))

# Reference fold for exploratory cells (TF-IDF inspection, UMAP, etc.)
ref_tr_idx, ref_va_idx = splits[0]
tr = train_df.iloc[ref_tr_idx].reset_index(drop=True)
va = train_df.iloc[ref_va_idx].reset_index(drop=True)
y_tr = y_all[ref_tr_idx]
y_va = y_all[ref_va_idx]

print('reference fold train split:', tr.shape)
print('reference fold valid split:', va.shape)


In [ ]:
# Structured feature preprocessing
structured_cols = infer_structured_columns(
    tr,
    target_col=target_col,
    text_col=text_col,
    id_col=id_col,
    explicit_drop_cols=leakage_cols,
)

structured_pre, x_tr_struct, x_va_struct = fit_transform_structured(tr[structured_cols], va[structured_cols])
print('structured feature count:', x_tr_struct.shape[1])

# Compare Text Representations

We compare lexical and semantic text representations to understand how complaint language contributes to triage acuity prediction.


## TF-IDF

TF-IDF captures discriminative words and short phrases. It is interpretable and serves as a strong baseline for chief-complaint modeling.


In [ ]:
import umap
import matplotlib.pyplot as plt

tfidf_vec, x_tr_tfidf, x_va_tfidf = fit_transform_tfidf(
    tr[text_col],
    va[text_col],
    max_features=25000,
    ngram_range=(1, 2),
)

print(x_tr_tfidf.shape, x_va_tfidf.shape)

In [ ]:
if RUN_PROJECTIONS:
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, n_components=2, metric='cosine', random_state=42)
    embedding = reducer.fit_transform(x_tr_tfidf)
    
    plt.figure(figsize=(10, 7))
    plt.scatter(embedding[:, 0], embedding[:, 1], s=10, alpha=0.7)
    plt.title("UMAP Projection of TF-IDF Features")
    plt.show()
else:
    print('UMAP projection skipped; set TRIAGE_PROJECTIONS=1 to run.')


## Frozen MiniLM embeddings

The pretrained encoder is not fine-tuned. Unique complaints are encoded once for efficiency; this does not fit on labels or validation data. All learned preprocessing and classifiers are fit inside each training fold.

## Baselines and fusion models

We train and compare:
1. Structured-only
2. Text-only (TF-IDF + linear model)
3. Structured + TF-IDF
4. Text-only pretrained embeddings
5. Structured + pretrained embeddings (Complaint2Concern fusion)

In [ ]:
# 1) Structured-only
# 2) Text-only TF-IDF + linear model
# 3) Structured + TF-IDF
# 4) Text-only pretrained embeddings
# 5) Structured + pretrained embeddings (main fusion)

classes_int = np.asarray(label_encoder.classes_).astype(int)
high_mask = classes_int <= 2
n = len(train_df)
n_classes = len(label_encoder.classes_)

emb_model_name = cfg['embeddings'].get('model_name', 'sentence-transformers/all-MiniLM-L6-v2')
emb_batch_size = cfg['embeddings'].get('batch_size', 64)
emb_backend = cfg['embeddings'].get('backend', 'sentence_transformers')

# OOF containers
oof = {
    'true_enc': y_all.copy(),
    'pred_structured': np.full(n, -1, dtype=int),
    'pred_text_tfidf': np.full(n, -1, dtype=int),
    'pred_structured_tfidf': np.full(n, -1, dtype=int),
    'pred_text_emb': np.full(n, -1, dtype=int),
    'pred_fusion': np.full(n, -1, dtype=int),
    'proba_structured': np.zeros((n, n_classes), dtype=float),
    'proba_text_emb': np.zeros((n, n_classes), dtype=float),
    'proba_fusion': np.zeros((n, n_classes), dtype=float),
}

# Interpretability containers (aggregated across folds)
interp_rows = []
interp_block_rows = []
tfidf_class_rows = []

cv_rows = []
ref_artifacts = {}

TOPK_GLOBAL = 100000
TOPK_CLASS_TERMS = 20

unique_texts, text_index = np.unique(train_df[text_col].to_numpy(), return_inverse=True)
unique_embeddings = encode_complaints(unique_texts, model_name=emb_model_name, batch_size=emb_batch_size)
all_embeddings = unique_embeddings[text_index]
print('Frozen encoder:', len(unique_texts), 'unique complaints for', len(train_df), 'rows')

for fold, (tr_idx, va_idx) in enumerate(splits, start=1):
    fold_start = time.time()

    tr_fold = train_df.iloc[tr_idx].reset_index(drop=True)
    va_fold = train_df.iloc[va_idx].reset_index(drop=True)
    y_tr_fold = y_all[tr_idx]
    y_va_fold = y_all[va_idx]

    struct_cols_fold = infer_structured_columns(
        tr_fold,
        target_col=target_col,
        text_col=text_col,
        id_col=id_col,
        explicit_drop_cols=leakage_cols,
    )
    struct_pre_fold, x_tr_struct_fold, x_va_struct_fold = fit_transform_structured(
        tr_fold[struct_cols_fold], va_fold[struct_cols_fold]
    )

    try:
        structured_feature_names = list(struct_pre_fold.get_feature_names_out())
    except Exception:
        structured_feature_names = [f'struct_{i}' for i in range(x_tr_struct_fold.shape[1])]

    # Structured-only
    structured_model = make_classifier(
        model_name='catboost',
        params={'iterations': 350, 'depth': 6, 'learning_rate': 0.05, 'verbose': False},
        n_classes=n_classes,
    )
    structured_model.fit(x_tr_struct_fold, y_tr_fold)
    pred_struct = np.asarray(structured_model.predict(x_va_struct_fold)).reshape(-1).astype(int)
    proba_struct = np.asarray(structured_model.predict_proba(x_va_struct_fold))

    # Structured-only importances
    try:
        fi_struct = np.asarray(structured_model.get_feature_importance())
        top_idx = np.argsort(fi_struct)[::-1][:TOPK_GLOBAL]
        for rank, idx in enumerate(top_idx, start=1):
            interp_rows.append({
                'fold': fold,
                'model': 'structured_only',
                'feature': structured_feature_names[idx],
                'importance': float(fi_struct[idx]),
                'rank': rank,
            })
    except Exception:
        pass

    # Text-only TF-IDF + LR
    tfidf_vec, x_tr_tfidf_fold, x_va_tfidf_fold = fit_transform_tfidf(
        tr_fold[text_col],
        va_fold[text_col],
        max_features=25000,
        ngram_range=(1, 2),
    )
    tfidf_model = LogisticRegression(max_iter=1200, solver='lbfgs')
    tfidf_model.fit(x_tr_tfidf_fold, y_tr_fold)
    pred_tfidf = tfidf_model.predict(x_va_tfidf_fold).astype(int)

    tfidf_feature_names = np.asarray(tfidf_vec.get_feature_names_out())

    # Text-only TF-IDF global importance + class terms
    coef_tfidf = np.asarray(tfidf_model.coef_)
    imp_tfidf = np.mean(np.abs(coef_tfidf), axis=0)
    top_idx = np.argsort(imp_tfidf)[::-1][:TOPK_GLOBAL]
    for rank, idx in enumerate(top_idx, start=1):
        interp_rows.append({
            'fold': fold,
            'model': 'text_tfidf_only',
            'feature': f"tfidf::{tfidf_feature_names[idx]}",
            'importance': float(imp_tfidf[idx]),
            'rank': rank,
        })

    for class_ix, class_label in enumerate(label_encoder.classes_):
        class_coef = coef_tfidf[class_ix]
        top_pos_idx = np.argsort(class_coef)[::-1][:TOPK_CLASS_TERMS]
        for rank, idx in enumerate(top_pos_idx, start=1):
            tfidf_class_rows.append({
                'fold': fold,
                'model': 'text_tfidf_only',
                'class_label': int(class_label),
                'term': tfidf_feature_names[idx],
                'coef': float(class_coef[idx]),
                'rank': rank,
            })

    # Structured + TF-IDF
    x_tr_struct_sparse = csr_matrix(x_tr_struct_fold)
    x_va_struct_sparse = csr_matrix(x_va_struct_fold)
    x_tr_struct_tfidf = hstack([x_tr_struct_sparse, x_tr_tfidf_fold]).tocsr()
    x_va_struct_tfidf = hstack([x_va_struct_sparse, x_va_tfidf_fold]).tocsr()
    struct_tfidf_model = LogisticRegression(max_iter=1200, solver='lbfgs')
    struct_tfidf_model.fit(x_tr_struct_tfidf, y_tr_fold)
    pred_struct_tfidf = struct_tfidf_model.predict(x_va_struct_tfidf).astype(int)

    mix_feature_names = np.array(structured_feature_names + [f"tfidf::{t}" for t in tfidf_feature_names])
    coef_mix = np.asarray(struct_tfidf_model.coef_)
    imp_mix = np.mean(np.abs(coef_mix), axis=0)
    top_idx = np.argsort(imp_mix)[::-1][:TOPK_GLOBAL]
    for rank, idx in enumerate(top_idx, start=1):
        interp_rows.append({
            'fold': fold,
            'model': 'structured_plus_tfidf',
            'feature': mix_feature_names[idx],
            'importance': float(imp_mix[idx]),
            'rank': rank,
        })

    # Structured + TF-IDF block importance
    s_dim = x_tr_struct_fold.shape[1]
    struct_mass = float(imp_mix[:s_dim].sum())
    tfidf_mass = float(imp_mix[s_dim:].sum())
    total_mass = struct_mass + tfidf_mass + 1e-12
    interp_block_rows.extend([
        {'fold': fold, 'model': 'structured_plus_tfidf', 'block': 'structured', 'importance': struct_mass, 'share': struct_mass / total_mass},
        {'fold': fold, 'model': 'structured_plus_tfidf', 'block': 'text_tfidf', 'importance': tfidf_mass, 'share': tfidf_mass / total_mass},
    ])

    # Text-only embeddings
    x_tr_emb_fold = all_embeddings[tr_idx]
    x_va_emb_fold = all_embeddings[va_idx]

    emb_text_model = make_classifier(
        model_name='lightgbm',
        params={'n_estimators': 350, 'learning_rate': 0.05, 'num_leaves': 63},
        n_classes=n_classes,
    )
    emb_text_model.fit(x_tr_emb_fold, y_tr_fold)
    pred_emb_text = np.asarray(emb_text_model.predict(x_va_emb_fold)).reshape(-1).astype(int)
    proba_emb_text = np.asarray(emb_text_model.predict_proba(x_va_emb_fold))

    emb_dim_names = np.array([f'emb_{i:03d}' for i in range(x_tr_emb_fold.shape[1])])
    try:
        imp_emb = np.asarray(emb_text_model.feature_importances_)
        top_idx = np.argsort(imp_emb)[::-1][:TOPK_GLOBAL]
        for rank, idx in enumerate(top_idx, start=1):
            interp_rows.append({
                'fold': fold,
                'model': 'text_emb_only',
                'feature': emb_dim_names[idx],
                'importance': float(imp_emb[idx]),
                'rank': rank,
            })
    except Exception:
        pass

    # Fusion model
    x_tr_fusion_fold = np.hstack([x_tr_struct_fold, x_tr_emb_fold])
    x_va_fusion_fold = np.hstack([x_va_struct_fold, x_va_emb_fold])
    fusion_model = make_classifier(
        model_name='catboost',
        params={'iterations': 450, 'depth': 6, 'learning_rate': 0.05, 'verbose': False},
        n_classes=n_classes,
    )
    fusion_model.fit(x_tr_fusion_fold, y_tr_fold)
    pred_fusion = np.asarray(fusion_model.predict(x_va_fusion_fold)).reshape(-1).astype(int)
    proba_fusion = np.asarray(fusion_model.predict_proba(x_va_fusion_fold))

    fusion_feature_names = np.array(structured_feature_names + [f'emb::{n}' for n in emb_dim_names])
    try:
        fi_fusion = np.asarray(fusion_model.get_feature_importance())
        top_idx = np.argsort(fi_fusion)[::-1][:TOPK_GLOBAL]
        for rank, idx in enumerate(top_idx, start=1):
            interp_rows.append({
                'fold': fold,
                'model': 'structured_plus_emb_fusion',
                'feature': fusion_feature_names[idx],
                'importance': float(fi_fusion[idx]),
                'rank': rank,
            })

        s_dim = x_tr_struct_fold.shape[1]
        struct_mass = float(fi_fusion[:s_dim].sum())
        emb_mass = float(fi_fusion[s_dim:].sum())
        total_mass = struct_mass + emb_mass + 1e-12
        interp_block_rows.extend([
            {'fold': fold, 'model': 'structured_plus_emb_fusion', 'block': 'structured', 'importance': struct_mass, 'share': struct_mass / total_mass},
            {'fold': fold, 'model': 'structured_plus_emb_fusion', 'block': 'text_embedding', 'importance': emb_mass, 'share': emb_mass / total_mass},
        ])
    except Exception:
        pass

    # Save OOF
    oof['pred_structured'][va_idx] = pred_struct
    oof['pred_text_tfidf'][va_idx] = pred_tfidf
    oof['pred_structured_tfidf'][va_idx] = pred_struct_tfidf
    oof['pred_text_emb'][va_idx] = pred_emb_text
    oof['pred_fusion'][va_idx] = pred_fusion
    oof['proba_structured'][va_idx] = proba_struct
    oof['proba_text_emb'][va_idx] = proba_emb_text
    oof['proba_fusion'][va_idx] = proba_fusion

    # Save one fold artifacts for local exploratory blocks
    if fold == 1:
        ref_artifacts = {
            'va': va_fold,
            'y_va': y_va_fold,
            'x_va_struct': x_va_struct_fold,
            'x_va_emb': x_va_emb_fold,
            'fusion_model': fusion_model,
            'proba_struct': proba_struct,
            'proba_emb_text': proba_emb_text,
            'pred_struct': pred_struct,
            'pred_emb_text': pred_emb_text,
            'pred_fusion': pred_fusion,
        }

    # Fold metrics
    def _eval_fold(y_true_enc, y_pred_enc):
        return evaluate_acuity(label_encoder.inverse_transform(y_true_enc), label_encoder.inverse_transform(y_pred_enc))

    fold_preds = {
        'structured_only': pred_struct,
        'text_tfidf_only': pred_tfidf,
        'structured_plus_tfidf': pred_struct_tfidf,
        'text_emb_only': pred_emb_text,
        'structured_plus_emb_fusion': pred_fusion,
    }

    for model_name, pred in fold_preds.items():
        cv_rows.append({
            'fold': fold,
            'model': model_name,
            **_eval_fold(y_va_fold, pred),
            'fold_runtime_sec': round(time.time() - fold_start, 2),
        })

    print(f'Fold {fold}/{len(splits)} complete in {round(time.time()-fold_start,2)}s')

# expose reference fold vars for exploratory cells below
va = ref_artifacts['va']
y_va = ref_artifacts['y_va']
x_va_struct = ref_artifacts['x_va_struct']
x_va_emb = ref_artifacts['x_va_emb']
fusion_model = ref_artifacts['fusion_model']
proba_struct = ref_artifacts['proba_struct']
proba_emb_text = ref_artifacts['proba_emb_text']
pred_struct = ref_artifacts['pred_struct']
pred_emb_text = ref_artifacts['pred_emb_text']
pred_fusion = ref_artifacts['pred_fusion']

# Interpretability dataframes available for downstream analysis cells
interp_df = pd.DataFrame(interp_rows)
interp_block_df = pd.DataFrame(interp_block_rows)
tfidf_class_df = pd.DataFrame(tfidf_class_rows)

print('CV training finished.')


In [ ]:
# 4) Text-only pretrained embeddings
# In CV mode, embeddings are computed fold-by-fold above.
# Here we just print reference fold embedding shapes for sanity checks.
print('Reference fold embedding shape:', x_va_emb.shape)
print('Embedding model:', emb_model_name)


In [ ]:
# ESI-aware evaluation helpers

def decode(enc):
    return label_encoder.inverse_transform(np.asarray(enc).astype(int))


def to_int_labels(labels):
    return np.asarray(labels).astype(int)


def off_by_one_acc(y_true_int, y_pred_int):
    return float(np.mean(np.abs(y_true_int - y_pred_int) <= 1))


def eval_esi(y_true_enc, y_pred_enc):
    return evaluate_acuity(decode(y_true_enc), decode(y_pred_enc))

cv_results_df = pd.DataFrame(cv_rows)
cv_summary = (
    cv_results_df
    .groupby('model', as_index=False)
    .agg(
        folds=('fold', 'count'),
        accuracy_mean=('accuracy', 'mean'),
        accuracy_std=('accuracy', 'std'),
        macro_f1_mean=('macro_f1', 'mean'),
        macro_f1_std=('macro_f1', 'std'),
        off_by_1_mean=('off_by_1', 'mean'),
        undertriage_mean=('undertriage_rate', 'mean'),
        #overtriage_mean=('overtriage_rate', 'mean'),
        severe_undertriage_mean=('severe_undertriage_rate', 'mean'),
        recall_esi_1_2_mean=('recall_esi_1_2', 'mean'),
    )
    .sort_values('macro_f1_mean', ascending=False)
)

# OOF aggregate performance
rows = [
    {'model': 'structured_only', **eval_esi(oof['true_enc'], oof['pred_structured'])},
    {'model': 'text_tfidf_only', **eval_esi(oof['true_enc'], oof['pred_text_tfidf'])},
    {'model': 'structured_plus_tfidf', **eval_esi(oof['true_enc'], oof['pred_structured_tfidf'])},
    {'model': 'text_emb_only', **eval_esi(oof['true_enc'], oof['pred_text_emb'])},
    {'model': 'structured_plus_emb_fusion', **eval_esi(oof['true_enc'], oof['pred_fusion'])},
]
perf_df = pd.DataFrame(rows).sort_values('macro_f1', ascending=False)

print('OOF performance table:')
display(perf_df)
print('CV mean +/- std summary:')
display(cv_summary)


In [ ]:
# CV stability summary (uses existing cv_summary; no retraining)
if 'cv_summary' not in globals():
    raise RuntimeError('cv_summary not found. Run the CV evaluation cell first.')

display_cols = [
    'model', 'folds',
    'macro_f1_mean', 'macro_f1_std',
    'accuracy_mean', 'accuracy_std',
    'undertriage_mean', 'severe_undertriage_mean', 'recall_esi_1_2_mean'
]

stable_df = cv_summary[display_cols].copy()
display(stable_df)

try:
    import matplotlib.pyplot as plt
    plot_df = stable_df.sort_values('macro_f1_mean', ascending=True)
    plt.figure(figsize=(8, 4))
    plt.errorbar(plot_df['macro_f1_mean'], plot_df['model'], xerr=plot_df['macro_f1_std'], fmt='o')
    plt.title('CV Macro-F1 Mean ± Std')
    plt.xlabel('macro_f1')
    plt.ylabel('model')
    plt.grid(alpha=0.2)
    plt.show()
except Exception:
    pass


## Model interpretability (cross-validated)

This section aggregates feature-importance signals across folds to reduce single-split bias.
We report model-level drivers, class-specific TF-IDF terms, and modality contributions for mixed models.


In [ ]:
# Global feature importance aggregated across folds (all models)
if 'interp_df' not in globals() or interp_df.empty:
    raise RuntimeError('interp_df not found. Run the main CV training cell first.')

global_importance_summary = (
    interp_df
    .groupby(['model', 'feature'], as_index=False)
    .agg(
        importance_mean=('importance', 'mean'),
        importance_std=('importance', 'std'),
        fold_count=('fold', 'nunique'),
        best_rank=('rank', 'min'),
    )
)

for model_name in global_importance_summary['model'].unique():
    print(f"Top features for {model_name} (mean importance across folds)")
    display(
        global_importance_summary[global_importance_summary['model'] == model_name]
        .sort_values('importance_mean', ascending=False)
        .head(20)
    )



In [ ]:
# Block-level contribution (structured vs text) for mixed models
if 'interp_block_df' not in globals() or interp_block_df.empty:
    print('No block-level importance available.')
else:
    block_summary = (
        interp_block_df
        .groupby(['model', 'block'], as_index=False)
        .agg(
            share_mean=('share', 'mean'),
            share_std=('share', 'std'),
        )
    )
    display(block_summary.sort_values(['model', 'share_mean'], ascending=[True, False]))



In [ ]:
# Class-specific TF-IDF terms (most positive coefficients per class)
if 'tfidf_class_df' not in globals() or tfidf_class_df.empty:
    print('No TF-IDF class-term data available.')
else:
    tfidf_terms_summary = (
        tfidf_class_df
        .groupby(['model', 'class_label', 'term'], as_index=False)
        .agg(
            coef_mean=('coef', 'mean'),
            coef_std=('coef', 'std'),
            fold_count=('fold', 'nunique'),
            best_rank=('rank', 'min'),
        )
    )

    for class_label in sorted(tfidf_terms_summary['class_label'].unique()):
        print(f"Top TF-IDF terms for class {class_label}")
        display(
            tfidf_terms_summary[tfidf_terms_summary['class_label'] == class_label]
            .sort_values('coef_mean', ascending=False)
            .head(20)
        )



### Embedding semantic anchors

Exploratory associations between high-activation embedding dimensions and complaint terms. These are not causal explanations or exact token attributions.

In [ ]:
# Map important embedding dimensions to representative words/phrases (reference fold)
from sklearn.feature_extraction.text import TfidfVectorizer

if 'interp_df' not in globals() or interp_df.empty:
    raise RuntimeError('interp_df not found. Run the main CV training cell first.')
if 'x_va_emb' not in globals() or 'va' not in globals():
    raise RuntimeError('Reference fold artifacts not found. Run the main CV training cell first.')

imp_emb = interp_df[
    interp_df['model'].isin(['text_emb_only', 'structured_plus_emb_fusion'])
].copy()

# Keep only embedding-like feature names
imp_emb = imp_emb[imp_emb['feature'].str.contains('emb', na=False)]
emb_global = (
    imp_emb.groupby('feature', as_index=False)
    .agg(importance_mean=('importance', 'mean'))
    .sort_values('importance_mean', ascending=False)
)

# Parse embedding dimension index from names like emb_012 or emb::emb_012 / emb::emb_12
pattern = re.compile(r'emb[_:]*0*([0-9]+)$')
def parse_dim(name: str):
    m = pattern.search(str(name))
    return int(m.group(1)) if m else None

emb_global['dim'] = emb_global['feature'].map(parse_dim)
emb_global = emb_global.dropna().astype({'dim': int})

TOP_DIMS = 10
TOP_Q = 0.98  # top activation quantile
TOP_TERMS = 12
TOP_EXAMPLES = 5

texts = va[text_col].astype(str).tolist()
vectorizer = TfidfVectorizer(max_features=15000, ngram_range=(1,2), stop_words='english')
X_txt = vectorizer.fit_transform(texts)
terms = np.asarray(vectorizer.get_feature_names_out())

anchor_rows = []
example_rows = []

for _, row in emb_global.head(TOP_DIMS).iterrows():
    d = int(row['dim'])
    if d >= x_va_emb.shape[1]:
        continue

    vals = x_va_emb[:, d]
    thr = np.quantile(vals, TOP_Q)
    idx = np.where(vals >= thr)[0]
    if len(idx) < 5:
        idx = np.argsort(vals)[-max(5, int(0.01 * len(vals))):]

    # Representative terms among high-activation complaints
    x_sub = X_txt[idx]
    term_scores = np.asarray(x_sub.mean(axis=0)).ravel()
    top_term_idx = np.argsort(term_scores)[::-1][:TOP_TERMS]
    top_terms = ', '.join(terms[top_term_idx])

    anchor_rows.append({
        'embedding_feature': row['feature'],
        'dim': d,
        'importance_mean': float(row['importance_mean']),
        'n_high_activation_cases': int(len(idx)),
        'anchor_terms': top_terms,
    })

    top_case_idx = idx[np.argsort(vals[idx])[::-1][:TOP_EXAMPLES]]
    for rank, i_case in enumerate(top_case_idx, start=1):
        example_rows.append({
            'embedding_dim': d,
            'rank': rank,
            'activation': float(vals[i_case]),
            'complaint_example': texts[i_case],
        })

embedding_anchor_df = pd.DataFrame(anchor_rows)
embedding_anchor_examples_df = pd.DataFrame(example_rows)

print('Embedding semantic anchors (high-activation term proxies):')
display(embedding_anchor_df)

print('Representative complaints for top embedding dimensions:')
display(embedding_anchor_examples_df.head(50))



Interpretation note: these anchors are **associations**, not exact token-level attributions.
They help translate dense embedding dimensions into clinically readable complaint themes.


## Cross-validation stability and uncertainty

Fold summaries and out-of-fold metrics are reported separately. Random stratified folds can share repeated synthetic complaint patterns; they do not establish generalisation to unseen wording or hospitals.

- macro-F1: balance across classes (important if imbalanced).
  - undertriage/severe undertriage: safety-critical.
  - recall ESI 1-2: ability to catch urgent patients.

The tables above report the current run. Compare text-only models with the combined model rather than assuming fusion improves prediction.

### Model comparison

The generated report exports exact values from this run. Historical metrics are documented separately in the Git history.

The historical competition workflow selected structured-plus-embedding fusion. That choice is retained for optional submission export; it is not a claim that fusion wins the internal comparison.

In [ ]:
# Confusion matrix (fusion, OOF)
y_true_int = to_int_labels(decode(oof['true_enc']))
y_pred_fusion_int = to_int_labels(decode(oof['pred_fusion']))
cm = pd.crosstab(
    pd.Series(y_true_int, name='true_acuity'),
    pd.Series(y_pred_fusion_int, name='pred_acuity'),
    dropna=False,
)
display(cm)


## Complaint-structure mismatch

We define a Text Concern Gap using high-acuity probability mass (ESI 1-2):

`gap = P_text(ESI 1-2) - P_structured(ESI 1-2)`

Positive values indicate text model is more alarmed than structured-only model.

In [ ]:
classes_int = np.asarray(label_encoder.classes_).astype(int)
high_mask = classes_int <= 2

p_high_text = oof['proba_text_emb'][:, high_mask].sum(axis=1)
p_high_struct = oof['proba_structured'][:, high_mask].sum(axis=1)
concern_gap = p_high_text - p_high_struct

mismatch_df = train_df[[id_col, text_col]].copy()
mismatch_df['true_acuity'] = y_true_int
mismatch_df['pred_structured'] = to_int_labels(decode(oof['pred_structured']))
mismatch_df['pred_text_emb'] = to_int_labels(decode(oof['pred_text_emb']))
mismatch_df['pred_fusion'] = y_pred_fusion_int
mismatch_df['p_high_text'] = p_high_text
mismatch_df['p_high_struct'] = p_high_struct
mismatch_df['text_concern_gap'] = concern_gap

gap_threshold = np.quantile(mismatch_df['text_concern_gap'], 0.90)
alert_cases = mismatch_df[mismatch_df['text_concern_gap'] >= gap_threshold].copy()

print('gap threshold (p90):', round(float(gap_threshold), 4))
print('alert case count:', len(alert_cases))
print('alert true high-acuity rate:', round((alert_cases['true_acuity'] <= 2).mean(), 4))

display(alert_cases.sort_values('text_concern_gap', ascending=False).head(20))


In [ ]:
# Concern-gap diagnostics (uses existing mismatch_df; no retraining)
if 'mismatch_df' not in globals():
    raise RuntimeError('mismatch_df not found. Run the mismatch cell first.')

q = mismatch_df['text_concern_gap'].quantile([0, 0.1, 0.25, 0.5, 0.75, 0.9, 0.99, 1.0])
print('Concern-gap quantiles:')
display(q.to_frame('text_concern_gap'))

mismatch_df['is_high_acuity'] = (mismatch_df['true_acuity'] <= 2).astype(int)
summary_gap = mismatch_df.groupby('is_high_acuity')['text_concern_gap'].describe(percentiles=[0.5, 0.9])[['mean', 'std', '50%', '90%']]
print('Gap by high-acuity (0=no, 1=yes):')
display(summary_gap)

try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(7, 3.5))
    mismatch_df['text_concern_gap'].hist(bins=80)
    plt.title('Distribution of Text Concern Gap')
    plt.xlabel('text_concern_gap')
    plt.ylabel('count')
    plt.show()
except Exception:
    pass


Concern-gap enrichment is descriptive. Its threshold is calculated on these out-of-fold predictions and has not been validated prospectively.

## Complaint clustering
- What it does: clusters embedding vectors and summarizes acuity mix by cluster.
- Why it matters: supports clinically interpretable complaint families.
- What to look for: high-acuity-enriched clusters and high concern-gap clusters.


In [ ]:
if RUN_PROJECTIONS:
    try:
        import umap
        import matplotlib.pyplot as plt
    except Exception as exc:
        raise ImportError("Install umap-learn to run this visualization: pip install umap-learn") from exc
    reducer = umap.UMAP(
        n_neighbors=15,
        min_dist=0.1,
        n_components=2,
        random_state=42
    )
    embedding = reducer.fit_transform(x_va_emb)
    # Convert encoded y_va to actual acuity levels (1..5)
    acuity_ref = label_encoder.inverse_transform(y_va).astype(int)
    # Categorical color map (fixed colors per class)
    class_colors = {
        1: "#d73027",  # red (most urgent)
        2: "#fc8d59",
        3: "#fee08b",
        4: "#91bfdb",
        5: "#4575b4",  # blue (least urgent)
    }
    plt.figure(figsize=(10, 7))
    for acuity in sorted(np.unique(acuity_ref)):
        mask = acuity_ref == acuity
        plt.scatter(
            embedding[mask, 0],
            embedding[mask, 1],
            s=10,
            alpha=0.7,
            c=class_colors.get(int(acuity), "#999999"),
            label=f"Acuity {acuity}",
        )
    plt.title("UMAP Projection of MiniLM Embeddings (categorical acuity colors)")
    plt.xlabel("UMAP Dimension 1")
    plt.ylabel("UMAP Dimension 2")
    plt.legend(title="Triage Acuity", loc="best", frameon=True)
    plt.show()
else:
    print('UMAP projection skipped; set TRIAGE_PROJECTIONS=1 to run.')


In [ ]:
# Complaint clustering and interpretation (reference fold)
y_true_ref_int = to_int_labels(decode(y_va))
concern_gap_ref = (
    ref_artifacts['proba_emb_text'][:, high_mask].sum(axis=1)
    - ref_artifacts['proba_struct'][:, high_mask].sum(axis=1)
)

n_clusters = 8
kmeans = KMeans(n_clusters=n_clusters, random_state=SEED, n_init=10)
cluster_id = kmeans.fit_predict(x_va_emb)

cluster_df = va[[text_col]].copy()
cluster_df['cluster'] = cluster_id
cluster_df['true_acuity'] = y_true_ref_int
cluster_df['text_concern_gap'] = concern_gap_ref

cluster_summary = cluster_df.groupby('cluster').agg(
    n=('cluster', 'size'),
    high_acuity_rate=('true_acuity', lambda s: np.mean(s <= 2)),
    mean_text_concern_gap=('text_concern_gap', 'mean'),
).sort_values('high_acuity_rate', ascending=False)
display(cluster_summary)

# Representative complaints: first few examples per cluster
examples = (
    cluster_df.groupby('cluster')[text_col]
    .apply(lambda s: list(s.head(5)))
    .to_frame('sample_complaints')
)
display(examples)

cluster_esi = pd.crosstab(cluster_df['cluster'], cluster_df['true_acuity'], normalize='index')
display(cluster_esi)


## Review-flag logic
- What it does: flags low-confidence or high disagreement cases.
- Why it matters: supports clinician review rather than autonomous triage override.
- What to look for: flagged subset enriched for higher acuity.


In [ ]:
# Review-flag logic (support tool, not auto-overrule)
pmax_fusion = oof['proba_fusion'].max(axis=1)
low_conf_threshold = np.quantile(pmax_fusion, 0.10)
high_gap_threshold = np.quantile(concern_gap, 0.90)

mismatch_df['fusion_confidence'] = pmax_fusion
mismatch_df['review_flag'] = (
    (mismatch_df['fusion_confidence'] <= low_conf_threshold)
    | (mismatch_df['text_concern_gap'] >= high_gap_threshold)
)

flagged = mismatch_df[mismatch_df['review_flag']]
print('review flag rate:', round(len(flagged) / len(mismatch_df), 4))
print('high-acuity rate among flagged:', round((flagged['true_acuity'] <= 2).mean(), 4))
display(flagged.head(20))


Review flags use empirical confidence and disagreement quantiles. Rates below describe this dataset, not a validated clinical alert policy.

In [ ]:
# Robustness quick checks: typo and truncation perturbations
def perturb_typo(s: str) -> str:
    t = '' if s is None else str(s)
    if len(t) < 5:
        return t
    return t[:-1] + 'x'


def perturb_short(s: str) -> str:
    t = '' if s is None else str(s)
    return ' '.join(t.split()[:2])

rob_n = min(2000, len(va))
rob_text = va[text_col].head(rob_n).tolist()
rob_typo = [perturb_typo(x) for x in rob_text]
rob_short = [perturb_short(x) for x in rob_text]

rob_emb = encode_complaints(rob_text, model_name=emb_model_name, batch_size=emb_batch_size, backend=emb_backend)
rob_typo_emb = encode_complaints(rob_typo, model_name=emb_model_name, batch_size=emb_batch_size, backend=emb_backend)
rob_short_emb = encode_complaints(rob_short, model_name=emb_model_name, batch_size=emb_batch_size, backend=emb_backend)

rob_struct = x_va_struct[:rob_n]
pred_base = np.asarray(fusion_model.predict(np.hstack([rob_struct, rob_emb]))).reshape(-1).astype(int)
pred_typo = np.asarray(fusion_model.predict(np.hstack([rob_struct, rob_typo_emb]))).reshape(-1).astype(int)
pred_short = np.asarray(fusion_model.predict(np.hstack([rob_struct, rob_short_emb]))).reshape(-1).astype(int)

print('stability vs typo perturbation:', round(float(np.mean(pred_base == pred_typo)), 4))
print('stability vs shortened complaint:', round(float(np.mean(pred_base == pred_short)), 4))


These are simple prediction-stability checks on up to 2,000 reference-fold cases. They do not establish clinical robustness.

In [ ]:
# Fairness / subgroup snapshot on OOF fusion predictions
subgroups = [c for c in ['age_group', 'sex', 'site_id'] if c in train_df.columns]
subgroup_rows = []
for col in subgroups:
    temp = train_df[[col]].copy()
    temp['y_true'] = y_true_int
    temp['y_pred'] = y_pred_fusion_int
    for grp, gdf in temp.groupby(col):
        if len(gdf) < 50:
            continue
        subgroup_rows.append({
            'group_col': col,
            'group_val': grp,
            'n': len(gdf),
            'accuracy': float(np.mean(gdf['y_true'] == gdf['y_pred'])),
            'undertriage_rate': float(np.mean(gdf['y_pred'] > gdf['y_true'])),
        })
subgroup_df = pd.DataFrame(subgroup_rows).sort_values(['group_col', 'n'], ascending=[True, False])
display(subgroup_df.head(30))


Subgroup metrics describe this synthetic dataset. Site-stratified summaries are distinct from holding out a site during training.

## Additional validation

Site-held-out and shuffled-text controls below are optional and disabled by default. Their implementation must not be reported as completed evaluation unless the current run executes them.

In [ ]:
# Shortcut-risk quick diagnostics (no retraining)
# 1) Complaint duplication ratio
complaint_counts = train_df[text_col].astype(str).value_counts()
dup_ratio = float((complaint_counts > 1).mean())
print('Unique complaint strings:', complaint_counts.shape[0])
print('Share of complaint strings appearing >1 time:', round(dup_ratio, 4))

# 2) Top complaints and acuity purity
top_n = 20
top_complaints = complaint_counts.head(top_n).index
purity_rows = []
for txt in top_complaints:
    sub = train_df[train_df[text_col].astype(str) == txt]
    mode_rate = sub[target_col].value_counts(normalize=True).iloc[0]
    purity_rows.append({'complaint': txt, 'n': len(sub), 'majority_acuity_share': float(mode_rate)})

purity_df = pd.DataFrame(purity_rows).sort_values('majority_acuity_share', ascending=False)
display(purity_df.head(10))


### Extra verification cells (optional but recommended)
The cells below implement checks from the checklist. They use existing OOF outputs where possible, so they do **not** require rerunning the full CV unless you explicitly enable the optional heavy checks.


In [ ]:
# 1) Undertriage direction sanity + per-class diagnostics (OOF, no retraining)
if 'oof' not in globals():
    raise RuntimeError('oof not found. Run CV training/evaluation cells first.')

# Direction sanity check: ESI 1 most urgent, ESI 5 least urgent
# Undertriage should be pred > true
y_true_int = decode(oof['true_enc']).astype(int)
y_pred_fusion_int = decode(oof['pred_fusion']).astype(int)
print('Direction sanity: undertriage = pred > true')
print('Undertriage rate (fusion OOF):', round(float((y_pred_fusion_int > y_true_int).mean()), 6))
print('Overtriage rate (fusion OOF):', round(float((y_pred_fusion_int < y_true_int).mean()), 6))

models_for_class = {
    'structured_only': oof['pred_structured'],
    'text_tfidf_only': oof['pred_text_tfidf'],
    'structured_plus_tfidf': oof['pred_structured_tfidf'],
    'text_emb_only': oof['pred_text_emb'],
    'structured_plus_emb_fusion': oof['pred_fusion'],
}

class_rows = []
classes = sorted(np.unique(y_true_int))
for model_name, pred_enc in models_for_class.items():
    pred_int = decode(pred_enc).astype(int)
    for c in classes:
        mask = (y_true_int == c)
        if mask.sum() == 0:
            continue
        recall_c = (pred_int[mask] == c).mean()
        # precision for class c
        pred_mask = (pred_int == c)
        precision_c = ((y_true_int[pred_mask] == c).mean() if pred_mask.sum() > 0 else np.nan)
        class_rows.append({
            'model': model_name,
            'acuity_class': c,
            'n_true': int(mask.sum()),
            'recall': float(recall_c),
            'precision': float(precision_c),
        })

per_class_df = pd.DataFrame(class_rows)
display(per_class_df.sort_values(['model','acuity_class']))

# High-acuity focused summary (ESI 1-2)
high_rows = []
for model_name, pred_enc in models_for_class.items():
    pred_int = decode(pred_enc).astype(int)
    high_mask_local = y_true_int <= 2
    high_rows.append({
        'model': model_name,
        'recall_esi_1_2': float((pred_int[high_mask_local] <= 2).mean()),
        'undertriage_within_esi_1_2': float((pred_int[high_mask_local] > y_true_int[high_mask_local]).mean()),
    })

display(pd.DataFrame(high_rows).sort_values('recall_esi_1_2', ascending=False))


In [ ]:
# 2) Confidence-vs-error diagnostics (OOF fusion, no retraining)
if 'oof' not in globals():
    raise RuntimeError('oof not found. Run CV training/evaluation cells first.')

conf = oof['proba_fusion'].max(axis=1)
correct = (oof['pred_fusion'] == oof['true_enc'])
y_true_int = decode(oof['true_enc']).astype(int)
y_pred_int = decode(oof['pred_fusion']).astype(int)
under = y_pred_int > y_true_int

diag_df = pd.DataFrame({
    'confidence': conf,
    'correct': correct,
    'undertriage': under,
    'true_acuity': y_true_int,
    'pred_acuity': y_pred_int,
})

summary = pd.DataFrame({
    'group': ['correct','incorrect','undertriage','not_undertriage'],
    'mean_confidence': [
        diag_df.loc[diag_df['correct'], 'confidence'].mean(),
        diag_df.loc[~diag_df['correct'], 'confidence'].mean(),
        diag_df.loc[diag_df['undertriage'], 'confidence'].mean(),
        diag_df.loc[~diag_df['undertriage'], 'confidence'].mean(),
    ]
})
display(summary)

# Confidence bins vs error rate
bins = np.linspace(0.0, 1.0, 11)
diag_df['conf_bin'] = pd.cut(diag_df['confidence'], bins=bins, include_lowest=True)
err_by_bin = diag_df.groupby('conf_bin', observed=False).agg(
    n=('confidence','size'),
    error_rate=('correct', lambda s: 1 - s.mean()),
    undertriage_rate=('undertriage','mean'),
)
display(err_by_bin)


In [ ]:
# 3) Calibration-style check for high-acuity risk (OOF fusion, no retraining)
# Event: high acuity (ESI 1-2)
if 'oof' not in globals():
    raise RuntimeError('oof not found. Run CV training/evaluation cells first.')

classes_int = np.asarray(label_encoder.classes_).astype(int)
high_mask = classes_int <= 2
p_high_fusion = oof['proba_fusion'][:, high_mask].sum(axis=1)
y_high_true = (decode(oof['true_enc']).astype(int) <= 2).astype(int)

cal_df = pd.DataFrame({'p_high': p_high_fusion, 'y_high': y_high_true})
cal_df['bin'] = pd.qcut(cal_df['p_high'], q=10, duplicates='drop')
cal_summary = cal_df.groupby('bin', observed=False).agg(
    n=('p_high','size'),
    mean_pred=('p_high','mean'),
    obs_rate=('y_high','mean'),
)
cal_summary['calibration_gap'] = cal_summary['mean_pred'] - cal_summary['obs_rate']
display(cal_summary)


In [ ]:
# 4) Rare-vs-common complaint performance (OOF fusion, no retraining)
if 'oof' not in globals():
    raise RuntimeError('oof not found. Run CV training/evaluation cells first.')

freq = train_df[text_col].astype(str).value_counts()
row_freq = train_df[text_col].astype(str).map(freq)

y_true_int = decode(oof['true_enc']).astype(int)
y_pred_int = decode(oof['pred_fusion']).astype(int)

freq_df = pd.DataFrame({
    'complaint_freq': row_freq.values,
    'correct': (oof['pred_fusion'] == oof['true_enc']).astype(int),
    'undertriage': (y_pred_int > y_true_int).astype(int),
})

freq_df['freq_band'] = pd.cut(
    freq_df['complaint_freq'],
    bins=[0, 1, 2, 5, 10, 20, np.inf],
    labels=['1', '2', '3-5', '6-10', '11-20', '20+'],
    include_lowest=True,
)

band_summary = freq_df.groupby('freq_band', observed=False).agg(
    n=('correct','size'),
    accuracy=('correct','mean'),
    undertriage_rate=('undertriage','mean'),
)
display(band_summary)


In [ ]:
# 5) Optional heavy check: Group-site CV for selected models
RUN_GROUP_SITE_CHECK = False  # set True to execute (will retrain models)

if RUN_GROUP_SITE_CHECK:
    if 'site_id' not in train_df.columns:
        raise RuntimeError('site_id not found; cannot run group-site CV check.')

    g_splits = list(GroupKFold(n_splits=N_SPLITS).split(train_df, y_all, groups=train_df['site_id']))
    rows_group = []

    for fold, (tr_idx, va_idx) in enumerate(g_splits, start=1):
        tr_fold = train_df.iloc[tr_idx].reset_index(drop=True)
        va_fold = train_df.iloc[va_idx].reset_index(drop=True)
        y_tr_fold = y_all[tr_idx]
        y_va_fold = y_all[va_idx]

        struct_cols = infer_structured_columns(
            tr_fold,
            target_col=target_col,
            text_col=text_col,
            id_col=id_col,
            explicit_drop_cols=leakage_cols,
        )
        _, x_tr_struct, x_va_struct = fit_transform_structured(tr_fold[struct_cols], va_fold[struct_cols])

        # Structured-only
        model_struct = make_classifier('catboost', {'iterations': 300, 'depth': 6, 'learning_rate': 0.05, 'verbose': False}, n_classes=len(label_encoder.classes_))
        model_struct.fit(x_tr_struct, y_tr_fold)
        pred_struct = np.asarray(model_struct.predict(x_va_struct)).reshape(-1).astype(int)

        # Text-only TF-IDF
        _, x_tr_tfidf, x_va_tfidf = fit_transform_tfidf(tr_fold[text_col], va_fold[text_col], max_features=25000, ngram_range=(1,2))
        model_tfidf = LogisticRegression(max_iter=1200, solver='lbfgs')
        model_tfidf.fit(x_tr_tfidf, y_tr_fold)
        pred_tfidf = model_tfidf.predict(x_va_tfidf).astype(int)

        # Fusion MiniLM
        x_tr_emb = encode_complaints(tr_fold[text_col], model_name=emb_model_name, batch_size=emb_batch_size)
        x_va_emb = encode_complaints(va_fold[text_col], model_name=emb_model_name, batch_size=emb_batch_size)
        model_fusion = make_classifier('catboost', {'iterations': 350, 'depth': 6, 'learning_rate': 0.05, 'verbose': False}, n_classes=len(label_encoder.classes_))
        model_fusion.fit(np.hstack([x_tr_struct, x_tr_emb]), y_tr_fold)
        pred_fusion = np.asarray(model_fusion.predict(np.hstack([x_va_struct, x_va_emb]))).reshape(-1).astype(int)

        for m, p in [('structured_only', pred_struct), ('text_tfidf_only', pred_tfidf), ('structured_plus_emb_fusion', pred_fusion)]:
            r = eval_esi(y_va_fold, p)
            rows_group.append({'fold': fold, 'model': m, **r})

    group_cv = pd.DataFrame(rows_group).groupby('model', as_index=False).agg(
        folds=('fold','count'),
        macro_f1_mean=('macro_f1','mean'),
        undertriage_mean=('undertriage_rate','mean'),
        recall_esi_1_2_mean=('recall_esi_1_2','mean'),
    ).sort_values('macro_f1_mean', ascending=False)

    print('Group-site CV summary:')
    display(group_cv)
else:
    print('Set RUN_GROUP_SITE_CHECK=True to run group-site CV comparison.')


In [ ]:
# 6) Optional heavy check: shuffled-text negative control
RUN_SHUFFLE_TEXT_CONTROL = False  # set True to execute (trains a control model)

if RUN_SHUFFLE_TEXT_CONTROL:
    rng = np.random.default_rng(SEED)

    # Use first CV fold for a fast control
    tr_idx, va_idx = splits[0]
    tr_fold = train_df.iloc[tr_idx].reset_index(drop=True)
    va_fold = train_df.iloc[va_idx].reset_index(drop=True)
    y_tr_fold = y_all[tr_idx]
    y_va_fold = y_all[va_idx]

    shuffled_text = tr_fold[text_col].sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    _, x_tr_tfidf_shuf, x_va_tfidf = fit_transform_tfidf(shuffled_text, va_fold[text_col], max_features=25000, ngram_range=(1,2))

    model_ctrl = LogisticRegression(max_iter=1200, solver='lbfgs')
    model_ctrl.fit(x_tr_tfidf_shuf, y_tr_fold)
    pred_ctrl = model_ctrl.predict(x_va_tfidf).astype(int)

    ctrl_metrics = eval_esi(y_va_fold, pred_ctrl)
    print('Shuffled-text control metrics (single fold):')
    print(ctrl_metrics)
else:
    print('Set RUN_SHUFFLE_TEXT_CONTROL=True to run shuffled-text negative control.')


## Optional prediction export

When enabled, average probabilities from a fold ensemble. Each model trains on its fold training partition; this is not a single model refit on all rows.

In [ ]:
if RUN_SUBMISSION:
    # Refit with CV fold ensemble for submission
    SELECTED_MODEL = 'structured_plus_emb_fusion'  # choose from compared models
    
    splits_sub = make_splitter(train_df, y_all, mode=CV_MODE, n_splits=N_SPLITS, seed=SEED)
    proba_test_accum = np.zeros((len(test_df), len(label_encoder.classes_)), dtype=float)
    
    for fold, (tr_idx, va_idx) in enumerate(splits_sub, start=1):
        tr_fold = train_df.iloc[tr_idx].reset_index(drop=True)
        y_tr_fold = y_all[tr_idx]
    
        struct_cols_fold = infer_structured_columns(
            tr_fold,
            target_col=target_col,
            text_col=text_col,
            id_col=id_col,
            explicit_drop_cols=leakage_cols,
        )
        struct_pre_fold, x_tr_struct_fold, x_test_struct_fold = fit_transform_structured(
            tr_fold[struct_cols_fold], test_df[struct_cols_fold]
        )
    
        if SELECTED_MODEL == 'structured_only':
            model = make_classifier('catboost', {'iterations': 350, 'depth': 6, 'learning_rate': 0.05, 'verbose': False}, n_classes=len(label_encoder.classes_))
            model.fit(x_tr_struct_fold, y_tr_fold)
            proba_test_fold = np.asarray(model.predict_proba(x_test_struct_fold))
    
        elif SELECTED_MODEL == 'text_tfidf_only':
            _, x_tr_tfidf_fold, x_test_tfidf = fit_transform_tfidf(tr_fold[text_col], test_df[text_col], max_features=25000, ngram_range=(1, 2))
            model = LogisticRegression(max_iter=1200, solver='lbfgs')
            model.fit(x_tr_tfidf_fold, y_tr_fold)
            proba_test_fold = np.asarray(model.predict_proba(x_test_tfidf))
    
        elif SELECTED_MODEL == 'structured_plus_tfidf':
            _, x_tr_tfidf_fold, x_test_tfidf = fit_transform_tfidf(tr_fold[text_col], test_df[text_col], max_features=25000, ngram_range=(1, 2))
            x_tr_mix = hstack([csr_matrix(x_tr_struct_fold), x_tr_tfidf_fold]).tocsr()
            x_test_mix = hstack([csr_matrix(x_test_struct_fold), x_test_tfidf]).tocsr()
            model = LogisticRegression(max_iter=1200, solver='lbfgs')
            model.fit(x_tr_mix, y_tr_fold)
            proba_test_fold = np.asarray(model.predict_proba(x_test_mix))
    
        elif SELECTED_MODEL == 'text_emb_only':
            x_tr_emb_fold = encode_complaints(tr_fold[text_col], model_name=emb_model_name, batch_size=emb_batch_size, backend=emb_backend)
            x_test_emb = encode_complaints(test_df[text_col], model_name=emb_model_name, batch_size=emb_batch_size, backend=emb_backend)
            model = make_classifier('lightgbm', {'n_estimators': 350, 'learning_rate': 0.05, 'num_leaves': 63}, n_classes=len(label_encoder.classes_))
            model.fit(x_tr_emb_fold, y_tr_fold)
            proba_test_fold = np.asarray(model.predict_proba(x_test_emb))
    
        elif SELECTED_MODEL == 'structured_plus_emb_fusion':
            x_tr_emb_fold = encode_complaints(tr_fold[text_col], model_name=emb_model_name, batch_size=emb_batch_size, backend=emb_backend)
            x_test_emb = encode_complaints(test_df[text_col], model_name=emb_model_name, batch_size=emb_batch_size, backend=emb_backend)
            x_tr_mix = np.hstack([x_tr_struct_fold, x_tr_emb_fold])
            x_test_mix = np.hstack([x_test_struct_fold, x_test_emb])
            model = make_classifier('catboost', {'iterations': 450, 'depth': 6, 'learning_rate': 0.05, 'verbose': False}, n_classes=len(label_encoder.classes_))
            model.fit(x_tr_mix, y_tr_fold)
            proba_test_fold = np.asarray(model.predict_proba(x_test_mix))
    
        else:
            raise ValueError(f'Unsupported SELECTED_MODEL: {SELECTED_MODEL}')
    
        proba_test_accum += proba_test_fold
        print(f'Submission fold {fold}/{len(splits_sub)} done')
    
    proba_test_mean = proba_test_accum / len(splits_sub)
    test_pred_enc = proba_test_mean.argmax(axis=1)
    test_pred = label_encoder.inverse_transform(test_pred_enc).astype(int)
    
    submission = pd.DataFrame({id_col: test_df[id_col], target_col: test_pred})
    
    out_dir = RUN_DIR
    out_dir.mkdir(parents=True, exist_ok=True)
    submission_path = out_dir / 'submission_notebook_cv.csv'
    from src.validation import validate_submission
    validate_submission(submission, test_df[id_col], id_col, target_col)
    submission.to_csv(submission_path, index=False)
    print('saved:', submission_path)
    display(submission.head())
else:
    print('Submission refit skipped; set TRIAGE_SUBMISSION=1 to run.')


## Interpretation and limits

This is a synthetic-data research prototype. External validation, unseen-complaint evaluation, stronger negative controls and independently selected alert thresholds remain future work. No deployment or clinical benefit has been demonstrated.

## Export results and provenance
Aggregate reports are suitable for review; row-level predictions and competition data stay under ignored `outputs/`.

In [ ]:
tables = {
    'oof_metrics': perf_df, 'fold_metrics': cv_results_df, 'cv_summary': cv_summary,
    'feature_importance': global_importance_summary, 'modality_importance': block_summary,
    'tfidf_terms': tfidf_terms_summary, 'embedding_anchors': embedding_anchor_df,
    'subgroups': subgroup_df, 'per_class': per_class_df, 'calibration': cal_summary.reset_index(),
    'complaint_frequency': band_summary.reset_index(),
}
summary_checks = {
    'review_flag_rate': float(mismatch_df['review_flag'].mean()),
    'high_acuity_share_flagged': float((flagged['true_acuity'] <= 2).mean()),
    'typo_prediction_stability': float(np.mean(pred_base == pred_typo)),
    'shortened_prediction_stability': float(np.mean(pred_base == pred_short)),
    'site_holdout_executed': bool(RUN_GROUP_SITE_CHECK),
    'shuffled_text_executed': bool(RUN_SHUFFLE_TEXT_CONTROL),
    'submission_exported': bool(RUN_SUBMISSION),
}
manifest = export_run(RUN_DIR, cfg, tables, summary_checks, oof, train_df,
                      sample_n=DEV_SAMPLE_N, n_splits=N_SPLITS, split_mode=CV_MODE)
print('Saved run:', RUN_DIR)
print('Manifest:', RUN_DIR / 'run_manifest.json')
